In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.neural_network import MLPRegressor

pd.set_option("display.precision", 4)

Load data and define inputs and outpus

In [2]:
col_names=['Frequency_Hz','Angle_of_attack_deg','Chord_length_m','Free_stream_velocity_m/s','Suction_side_displacement_thickness_m','Scaled_sound_pressure_level_dB']
df = pd.read_csv(
    "../data/airfoil_self_noise.dat",
    sep=r"\s+",
    header=None,
    names=col_names
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
print(df.head())

X = df.iloc[:, 0:5]
y = df.iloc[:, [5]]



   Frequency_Hz  Angle_of_attack_deg  Chord_length_m  Free_stream_velocity_m/s  Suction_side_displacement_thickness_m  Scaled_sound_pressure_level_dB
0           800                  0.0          0.3048                      71.3                                 0.0027                         126.201
1          1000                  0.0          0.3048                      71.3                                 0.0027                         125.201
2          1250                  0.0          0.3048                      71.3                                 0.0027                         125.951
3          1600                  0.0          0.3048                      71.3                                 0.0027                         127.591
4          2000                  0.0          0.3048                      71.3                                 0.0027                         127.461


Data cleaning

In [3]:
#Check the basic structure of data
print("Shape: ")
print("DataFrame shape: ",df.shape)
print("X shape: ",X.shape)
print("Y shape: ",y.shape)
print("------------------------------------------")
#Check data type for each entry
print("Data type:\n",df.dtypes,sep="")
print("------------------------------------------")
#Chek missing values
print("Missing value count:\n",df.isnull().sum(),sep="")
print("------------------------------------------")
#Check duplicated rows
print("Duplicated rows:",df.duplicated().sum())
print("------------------------------------------")
#Check the overall statistics of data
print(df.describe())

Shape: 
DataFrame shape:  (1503, 6)
X shape:  (1503, 5)
Y shape:  (1503, 1)
------------------------------------------
Data type:
Frequency_Hz                               int64
Angle_of_attack_deg                      float64
Chord_length_m                           float64
Free_stream_velocity_m/s                 float64
Suction_side_displacement_thickness_m    float64
Scaled_sound_pressure_level_dB           float64
dtype: object
------------------------------------------
Missing value count:
Frequency_Hz                             0
Angle_of_attack_deg                      0
Chord_length_m                           0
Free_stream_velocity_m/s                 0
Suction_side_displacement_thickness_m    0
Scaled_sound_pressure_level_dB           0
dtype: int64
------------------------------------------
Duplicated rows: 0
------------------------------------------
       Frequency_Hz  Angle_of_attack_deg  Chord_length_m  Free_stream_velocity_m/s  Suction_side_displacement_thickness_m 

Split the data into train, validation and test set using group split becasue the experiment is about obtaining a spectrum of frequency, then analyze their spl, for different type of NACA0012 sizes and angle of attack. 

In [4]:
categorical_feature = ['Angle_of_attack_deg','Chord_length_m','Free_stream_velocity_m/s','Suction_side_displacement_thickness_m']
#Take the value of the categorical_feature column and convert them into a tuple. Assign each unique tuple (unique) with an ID (groups).
groups, unique = pd.factorize(X[categorical_feature].apply(tuple, axis=1))

#Split data into train, validation and test set, ratio of 60:20:20
#Use Group Split to split data once into train+validation and test set, ratio of 80:20
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
#Split the data and obtain the row idx for train_validation and test set
train_val_idx, test_idx = next(gss_test.split(X, y, groups=groups))

#Extract the data to their corresponding idx
X_train_val, X_test = X.iloc[train_val_idx], X.iloc[test_idx]
y_train_val, y_test = y.iloc[train_val_idx], y.iloc[test_idx]

#Use same procedure split train and validation set, ratio of 25:75
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
#Split the data and obtain the row idx for train and validation set
train_idx, val_idx = next(gss_val.split(X_train_val, y_train_val, groups=groups[train_val_idx]))
#Extract the data to their corresponding idx
X_train, X_val = X_train_val.iloc[train_idx], X_train_val.iloc[val_idx]
y_train, y_val= y_train_val.iloc[train_idx], y_train_val.iloc[val_idx]

Fit Linear Regression to fit data

In [ ]:
#Train Linear Regression model
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

#Report the intercept and coefficient in a data frame
coefficient_table = pd.DataFrame({
    "parameter": ['intercept', 'Frequency', 'Angle_of_attack_deg','Chord_length_m','Free_stream_velocity_m/s','Suction_side_displacement_thickness_m'],
    "value": [linear_model.intercept_.item()] + list(linear_model.coef_.ravel()),
    "units": ["dB", "dB/Hz", "dB/deg", "dB/m", "dBs/m", "dB/m"],
})

display(coefficient_table)


#Make prediction on validation set
val_pred_lr = linear_model.predict(X_val)
validation_results = X_val.copy()
#calculate and observe residuals
validation_results["observed_Scaled_sound_pressure_level_dB"] = y_val
validation_results["predicted_Scaled_sound_pressure_level_dB"] = val_pred_lr
validation_results["residual_dB"] = (
    validation_results["observed_Scaled_sound_pressure_level_dB"]
    - validation_results["predicted_Scaled_sound_pressure_level_dB"]
)

display(validation_results)

# Validation metrics and a baseline
def regression_metrics(y_true, y_pred):
    # Return common regression metrics in a labelled dictionary.
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE_dB": mean_absolute_error(y_true, y_pred),
        "RMSE_dB": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred),
    }


baseline_prediction = np.full(
    shape=len(y_val),
    fill_value=y_train.mean(),
)

metric_comparison = pd.DataFrame({
    "linear regression": regression_metrics(
        y_val,
        val_pred_lr,
    ),
    "mean baseline": regression_metrics(
        y_val,
        baseline_prediction,
    ),
}).T



,parameter,value,units
0,intercept,131.1134,dB
1,Frequency,-0.0013,dB/Hz
2,Angle_of_attack_deg,-0.3314,dB/deg
3,Chord_length_m,-32.5883,dB/m
4,Free_stream_velocity_m/s,0.1158,dBs/m
5,Suction_side_displacement_thickness_m,-147.0578,dB/m


,Frequency_Hz,Angle_of_attack_deg,Chord_length_m,Free_stream_velocity_m/s,Suction_side_displacement_thickness_m,observed_Scaled_sound_pressure_level_dB,predicted_Scaled_sound_pressure_level_dB,residual_dB
14,500,0.0,0.3048,55.5,0.0028,126.416,126.5437,-0.1277
15,630,0.0,0.3048,55.5,0.0028,127.696,126.3747,1.3213
16,800,0.0,0.3048,55.5,0.0028,128.086,126.1537,1.9323
17,1000,0.0,0.3048,55.5,0.0028,126.966,125.8938,1.0722
18,1250,0.0,0.3048,55.5,0.0028,126.086,125.5688,0.5172
...,...,...,...,...,...,...,...,...
1468,2500,12.3,0.1016,31.7,0.0419,110.317,117.9901,-7.6731
1469,3150,12.3,0.1016,31.7,0.0419,110.307,117.1451,-6.8381
1470,4000,12.3,0.1016,31.7,0.0419,108.407,116.0402,-7.6332
1471,5000,12.3,0.1016,31.7,0.0419,107.147,114.7402,-7.5932


,MAE_dB,RMSE_dB,R2
linear regression,3.8189,4.8170,0.5019
mean baseline,5.5165,6.8255,-0.0001


Use Neural Network Regression to fit data

In [ ]:
def make_mlp(random_state=42, hidden_layer_sizes=(20,), alpha=0.001,
             early_stopping=True):
    return MLPRegressor(
        hidden_layer_sizes=hidden_layer_sizes,
        activation="relu",
        solver="adam",
        alpha=alpha,
        learning_rate_init=0.01,
        max_iter=2500,
        early_stopping=early_stopping,
        validation_fraction=0.15,
        n_iter_no_change=35,
        tol=1e-5,
        random_state=random_state,
    )

#Define dictionary placeholder to place data of unscaled and scaled MLP
scaling_models = {
    "Unscaled MLP": make_mlp(),
    "Scaled MLP pipeline": make_pipeline(StandardScaler(), make_mlp()),
}

#Define a placeholder to store scaling models
scaling_rows = []

#Define metrics function
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def metrics(y_true, y_pred):
    return {
        "MAE_dB": mean_absolute_error(y_true, y_pred),
        "RMSE_dB": rmse(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }

# Calculate performance data for scaled and unscaled MLP
for label, model in scaling_models.items():
    model.fit(X_train, y_train)
    scaling_rows.append({"model": label,
                         **metrics(y_val, model.predict(X_val))})

#
scaling_results = pd.DataFrame(scaling_rows).set_index("model")


c:\Users\User\anaconda3\envs\mma3001\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:1775: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\User\anaconda3\envs\mma3001\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:1775: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,MAE_dB,RMSE_dB,R2
model,,,
Unscaled MLP,16.5959,19.8170,-7.4307
Scaled MLP pipeline,2.8957,4.0437,0.6490


Comparison of Mean, Linear Regression and Neural Network Regression validation metrics.

In [7]:
metric_comparison_overall = {}

metric_comparison_overall = pd.DataFrame({
    "mean baseline": regression_metrics(
            y_val,
            baseline_prediction,
    ),
    "linear regression": regression_metrics(
        y_val,
        val_pred_lr,
    ),
}).T

metric_comparison_overall = pd.concat([
    metric_comparison_overall,
    scaling_results
])

display(metric_comparison_overall)

,MAE_dB,RMSE_dB,R2
mean baseline,5.5165,6.8255,-0.0001
linear regression,3.8189,4.8170,0.5019
Unscaled MLP,16.5959,19.8170,-7.4307
Scaled MLP pipeline,2.8957,4.0437,0.6490
